# Notebook 01 — Multi-City Data Ingestion & Merge
## AeroTwinML · Hyderabad + Karachi, Pakistan

**Objective:** Fetch raw air quality and weather data for **two cities** (Hyderabad & Karachi),
normalize, validate, merge, and produce a combined hourly training table tagged with `city`.

**Data sources per city:**
| Source | Hyderabad | Karachi | Role |
|--------|-----------|---------|------|
| Open-Meteo | lat=25.396, lon=68.357 | lat=24.868, lon=67.082 | Weather features (predictors) |
| OpenAQ | station 4889110 | station 4791924 | Observed AQI/PM labels (ground truth) |
| AQICN | station A546205 | N/A | Secondary observed labels |

**Key principle:** More cities = more labeled training data = better model generalization.

In [ ]:
import sys, os, json
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta

from utils.config import get, all_config
from utils.time_utils import now_local, format_iso
from utils.storage import save_parquet

plt.style.use('dark_background')
sns.set_palette('viridis')
%matplotlib inline

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# Load locations config
locations = get('locations', [])
print(f'Configured cities: {[loc["name"] for loc in locations]}')
print(f'Timezone: {get("city.timezone")}')
print(f'Current local time: {format_iso(now_local())}')

## 1. Fetch Data for Each City

We create location-specific providers for each city and fetch data independently.

In [ ]:
from ingestion.providers.openmeteo import OpenMeteoProvider
from ingestion.providers.openaq import OpenAQProvider
from ingestion.providers.aqicn import AQICNProvider
from ingestion.orchestrator import IngestionOrchestrator

orch = IngestionOrchestrator()
city_data = {}

for loc in locations:
    city = loc['name']
    print(f'\n{"="*60}')
    print(f'  Fetching: {city} ({loc["latitude"]}, {loc["longitude"]})')
    print(f'  OpenAQ: {loc["openaq_location_id"]} | AQICN: {loc.get("aqicn_station", "N/A")}')
    print(f'{"="*60}')
    
    # Create city-specific providers
    om = OpenMeteoProvider(lat=loc['latitude'], lon=loc['longitude'], city_name=city)
    oaq = OpenAQProvider(location_id=loc['openaq_location_id'], city_name=city)
    aqicn = AQICNProvider(
        station=str(loc.get('aqicn_station') or ''),
        city_name=city
    )
    
    # Fetch Open-Meteo (weather + AQ forecast)
    try:
        raw_om = om.fetch_raw()
        om_df = om.validate(om.normalize(raw_om))
        print(f'  Open-Meteo: {len(om_df)} rows, {len(om_df.columns)} cols')
    except Exception as e:
        print(f'  Open-Meteo FAILED: {e}')
        om_df = pd.DataFrame()
    
    # Fetch OpenAQ (observed labels)
    try:
        raw_oaq = oaq.fetch_raw()
        oaq_df = oaq.validate(oaq.normalize(raw_oaq))
        print(f'  OpenAQ: {len(oaq_df)} rows')
    except Exception as e:
        print(f'  OpenAQ FAILED: {e}')
        oaq_df = pd.DataFrame()
    
    # Fetch AQICN (secondary labels, may be empty for Karachi)
    try:
        raw_aq = aqicn.fetch_raw()
        aq_df = aqicn.validate(aqicn.normalize(raw_aq))
        print(f'  AQICN: {len(aq_df)} rows')
    except Exception as e:
        print(f'  AQICN skipped: {e}')
        aq_df = pd.DataFrame()
    
    # Merge for this city
    observed_dfs = [df for df in [oaq_df, aq_df] if not df.empty]
    if not om_df.empty:
        merged_city = orch.merge(om_df, observed_dfs)
        merged_city['city'] = city
        city_data[city] = merged_city
        print(f'  Merged: {len(merged_city)} rows, AQI labels: {merged_city["aqi"].notna().sum() if "aqi" in merged_city.columns else 0}')
    else:
        print(f'  No data for {city}')

## 2. Combine All Cities

In [ ]:
# Combine all cities into one DataFrame
if city_data:
    combined = pd.concat(city_data.values(), ignore_index=True)
    combined = combined.sort_values(['city', 'timestamp']).reset_index(drop=True)
    
    print(f'Combined DataFrame: {combined.shape[0]} rows x {combined.shape[1]} columns')
    print(f'Cities: {combined["city"].unique().tolist()}')
    print(f'\nRows per city:')
    print(combined.groupby('city').size().to_string())
    print(f'\nAQI labels per city:')
    if 'aqi' in combined.columns:
        print(combined.groupby('city')['aqi'].apply(lambda x: x.notna().sum()).to_string())
else:
    print('No data fetched!')
    combined = pd.DataFrame()

## 3. Compare Cities Side-by-Side

In [ ]:
if not combined.empty:
    # Compare weather between cities
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    for ax, col, title in [
        (axes[0, 0], 'temperature_2m', 'Temperature (C)'),
        (axes[0, 1], 'relative_humidity_2m', 'Humidity (%)'),
        (axes[1, 0], 'wind_speed_10m', 'Wind Speed (m/s)'),
        (axes[1, 1], 'om_forecast_aqi', 'Open-Meteo Forecast AQI'),
    ]:
        if col in combined.columns:
            for city in combined['city'].unique():
                city_df = combined[combined['city'] == city]
                ax.plot(city_df['timestamp'], city_df[col], label=city, alpha=0.8, linewidth=0.8)
            ax.set_title(title)
            ax.legend()
            ax.grid(True, alpha=0.2)
    
    plt.suptitle('Hyderabad vs Karachi — Weather Comparison', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

## 4. Data Quality Summary

In [ ]:
if not combined.empty:
    print('=== Data Quality Summary ===')
    for city in combined['city'].unique():
        city_df = combined[combined['city'] == city]
        print(f'\n--- {city} ---')
        print(f'  Rows: {len(city_df)}')
        print(f'  Time range: {city_df["timestamp"].min()} to {city_df["timestamp"].max()}')
        aqi_count = city_df['aqi'].notna().sum() if 'aqi' in city_df.columns else 0
        print(f'  Observed AQI labels: {aqi_count} ({aqi_count/len(city_df)*100:.1f}%)')
        print(f'  Temperature range: {city_df["temperature_2m"].min():.1f} to {city_df["temperature_2m"].max():.1f} C')
        print(f'  Humidity range: {city_df["relative_humidity_2m"].min():.0f}% to {city_df["relative_humidity_2m"].max():.0f}%')

## 5. Save Combined Data

In [ ]:
if not combined.empty:
    # Save combined multi-city data
    save_path = Path('../data/processed/merged_hourly/merged_latest.parquet')
    save_path.parent.mkdir(parents=True, exist_ok=True)
    save_parquet(combined, save_path)
    print(f'Saved: {save_path} ({len(combined)} rows)')
    
    # Show final schema
    print(f'\nSchema:')
    for col in combined.columns:
        dtype = combined[col].dtype
        non_null = combined[col].notna().sum()
        print(f'  {col:30s} {str(dtype):15s} {non_null}/{len(combined)} non-null')

---
## Summary

| Step | What happened |
|------|--------------|
| 1. Per-city fetch | Open-Meteo + OpenAQ + AQICN fetched independently for each city |
| 2. Per-city merge | Weather features + observed labels merged on hourly timestamp |
| 3. Combine | All cities concatenated with `city` column tag |
| 4. Quality check | Verified data completeness, label coverage, value ranges |
| 5. Save | Persisted combined multi-city data to parquet |

**Next:** Notebook 01b — EDA comparing Hyderabad vs Karachi